In [0]:
from typing import Any

In [0]:
import logging

from py4j.java_gateway import JavaObject  # type: ignore
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col,
    concat_ws,
    current_timestamp,
    from_utc_timestamp,
    lit,
    sha2,
)

In [0]:
dbutils.widgets.text('curated_zone', 'None', '01-Curated_zone')
dbutils.widgets.text('catalog_landed', 'None', '02-Catalog_landed')
dbutils.widgets.text('reference_zone', 'None', '03-Reference_zone')
dbutils.widgets.text('run_date', 'None', '04-Run Date')
dbutils.widgets.text('schema', 'None', '05-Schema')
dbutils.widgets.text('env_prefix', 'None', '06-Environment')
curated_zone = dbutils.widgets.get('curated_zone')
catalog_landed = dbutils.widgets.get('catalog_landed')
reference_zone = dbutils.widgets.get('reference_zone')
run_date = dbutils.widgets.get('run_date')
schema = dbutils.widgets.get('schema')
env_prefix = dbutils.widgets.get('env_prefix')

import sys, os
relative_path = '../../helpers/lng_lnd'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from landing_helpers import *
from landed_helpers import *
relative_path = '../../helpers/curated_helpers'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from delta_table_creator import *
from extract import *
from transform import *
spark = SparkSession.builder.getOrCreate()


In [0]:
def load_curated_real_estate_info_thesis(
    spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix
) -> None:
    args = [curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix]
    if all(item is not None for item in args) :
        date_filter = f"businessdate = {run_date!r}"

        # schema = (
        #     "RealEstateInfoId String, SourceSystem String,"
        #     "IngestionTimestamp timestamp, BatchId String, RealEstateInfoKey string, RealEstateKey String, OwnerPartyKey String, OwnershipKey String, OwnershipReferenceKey String,"
        #     "OwnershipCode String, OwnershipTypeCode String, OwnershipType String, OwnershipPercentage Double,"
        #     "OwnershipStatusCode String, OwnershipStatus String, OwnershipStatusDate Date, NotaryDate Date, NotaryNumber String,"
        #     "RELienKey String, RELienReferenceKey String, RELienIsActiveFlag Boolean, RELienStatusCode String, RELienStatus String,"
        #     "RELienActive Boolean, RELienTypeCode String, RELienType String, RELienReasonCode String,"
        #     "RELienReason String, RELienInitialOrder Int, RELienCurrentOrder Int, RELienCreditorKey String, RELienCreditorName String,"
        #     "RELienCreditorType string, RELienMortgageTypeCode String, RELienMortgageType String, RELienMortgageAmount Decimal(19,4), "
        #     "RELienCurrency String, RELienComments String, BusinessDate date"
        # )

        cluster_columns = ["BusinessDate"]

        curated_table_path = f"{curated_zone}.RealEstateInfo"

        # ================== Create Delta Table ==================
        logging.info("Creating Delta Table...")
        if spark.catalog.tableExists(curated_table_path) is False:
            create_delta_table(
                spark,
                curated_table_path,
                cluster_columns,
                schema,  # type: ignore
            )
        else:
            logging.info("Table already exists.")

        # ================== Delete From Delta Table ==================
        if spark.catalog.tableExists(curated_table_path) is True:
            delete_from_curated_table(
                spark,
                curated_table_path,
                run_date,  # type: ignore
                "thesis",
            )        
        else:
            logging.info("Tables does not exist, no need to delete entries for business date!")

        # ================== prproperty_lien ==================

        prproperty_lien_extra_filters = None

        if prproperty_lien_extra_filters is not None:
            prproperty_lien_extra_filters.append(date_filter)
            prproperty_lien_final_filters = " AND ".join(prproperty_lien_extra_filters)
        else:
            prproperty_lien_final_filters = date_filter

        prproperty_lien_selected_fields = [
            "id",
            "ownership_id",
            "reference_key",
            "is_active",
            "collateral_status",
            "type_code",
            "reason_code",
            "initial_order",
            "current_order",
            "creditor_id",
            "mortgage_type",
            "amount",
            "curr_code",
            "comments",
        ]

        prproperty_lien = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "thesis_prproperty_lien",
            prproperty_lien_final_filters,
            prproperty_lien_selected_fields,
        )
        prproperty_lien = prproperty_lien.withColumnRenamed("id", "prproperty_lien_id")
        # ================== prproperty_owner ==================

        prproperty_owner_extra_filters = None

        if prproperty_owner_extra_filters is not None:
            prproperty_owner_extra_filters.append(date_filter)
            prproperty_owner_final_filters = " AND ".join(
                prproperty_owner_extra_filters
            )
        else:
            prproperty_owner_final_filters = date_filter

        prproperty_owner_selected_fields = [
            "id",
            "property_id",
            "person_id",
            "reference_key",
            "ownership_code",
            "realtitle_descr",
            "percentage",
            "ownership_status",
            "status_date",
            "notary_date",
            "notary_no",
            "businessdate",
        ]

        prproperty_owner = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "thesis_prproperty_owner",
            prproperty_owner_final_filters,
            prproperty_owner_selected_fields,
        )

        prproperty_owner = prproperty_owner.withColumn("prproperty_owner_id", col("id"))

        # ==============================thesis_prpropertytemp====================
        thesis_prpropertyownertemp_extra_filters = None

        if thesis_prpropertyownertemp_extra_filters is not None:
            thesis_prpropertyownertemp_extra_filters.append(date_filter)
            thesis_prpropertyownertemp_final_filters = " AND ".join(
                thesis_prpropertyownertemp_extra_filters
            )
        else:
            thesis_prpropertyownertemp_final_filters = date_filter


        thesis_prpropertyownertemp_selected_fields = [
            'id', 
            'property_temp_id', 
            'person_id', 
            'temp_ownership_id', 
            'ownership_id', 
            'realtitle_id', 
            'percentage', 
            'notary_date', 
            'notary_no',
            "BusinessDate",
        ]

        thesis_prpropertyownertemp = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "thesis_prpropertyownertemp",
            thesis_prpropertyownertemp_final_filters,
            thesis_prpropertyownertemp_selected_fields,
        )

        # ==============================thesis_prpropertybuildingtemp====================

        thesis_prpropertylientemp_extra_filters = None

        if thesis_prpropertylientemp_extra_filters is not None:
            thesis_prpropertylientemp_extra_filters.append(date_filter)
            thesis_prpropertylientemp_final_filters = " AND ".join(
                thesis_prpropertylientemp_extra_filters
            )
        else:
            thesis_prpropertylientemp_final_filters = date_filter

        thesis_prpropertylientemp_selected_fields = [
            'id', 
            'lien_id', 
            'subcat_code', 
            'initial_order',
            'creditor_id', 
            'mortgage_type', 
            'amount', 
            'curr_code', 
            'comments',
            "BusinessDate"
        ]

        thesis_prpropertylientemp = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "thesis_prpropertylientemp",
            thesis_prpropertylientemp_final_filters,
            thesis_prpropertylientemp_selected_fields,
        )
        
        # ================== Joins ==================

        prproperty_owner_prproperty_lien_join_condition = (
            prproperty_lien["ownership_id"] == prproperty_owner["id"]
        )

        target_table = prproperty_owner.join(
            prproperty_lien, prproperty_owner_prproperty_lien_join_condition, "left"
        )

        target_table_primary_key_fields = [
            target_table.prproperty_owner_id,
            target_table.prproperty_lien_id,
            lit("thesis"),
        ]
        RealEstateInfoKey_primary_key_fields = [
            target_table.prproperty_owner_id,
            target_table.prproperty_lien_id,
        ]

        target_table = target_table.withColumn(
            "RealEstateInfoId",
            sha2(concat_ws(";", *target_table_primary_key_fields), 256),
        ).withColumn(
            "RealEstateInfoKey", concat_ws(";", *RealEstateInfoKey_primary_key_fields)
        )

        result_df_thesis = target_table.select(
            target_table.RealEstateInfoId,
            lit("thesis").alias("SourceSystem"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens")).alias(
                "IngestionTimestamp"
            ),
            lit(None).cast("String").alias("BatchId"),
            target_table.RealEstateInfoKey.cast("string"),
            prproperty_owner.property_id.cast("String").alias("RealEstateKey"),
            prproperty_owner.person_id.cast("String").alias("OwnerPartyKey"),
            prproperty_owner.id.cast("string").alias("OwnershipKey"),
            prproperty_owner.reference_key.cast("string").alias(
                "OwnershipReferenceKey"
            ),
            prproperty_owner.ownership_code.alias("OwnershipCode"),
            prproperty_owner.realtitle_descr.alias("OwnershipTypeCode"),
            prproperty_owner.realtitle_descr.alias("OwnershipType"),
            prproperty_owner.percentage.alias("OwnershipPercentage"),
            prproperty_owner.ownership_status.alias("OwnershipStatusCode"),
            prproperty_owner.ownership_status.alias("OwnershipStatus"),
            prproperty_owner.status_date.alias("OwnershipStatusDate"),
            prproperty_owner.notary_date.alias("NotaryDate"),
            prproperty_owner.notary_no.alias("NotaryNumber"),
            prproperty_lien.prproperty_lien_id.cast("string").alias("RELienKey"),
            prproperty_lien.reference_key.cast("string").alias("RELienReferenceKey"),
            prproperty_lien.is_active.alias("RELienIsActiveFlag"),
            prproperty_lien.collateral_status.alias("RELienStatusCode"),
            prproperty_lien.collateral_status.alias("RELienStatus"),
            prproperty_lien.is_active.alias("RELienActive"),
            prproperty_lien.type_code.alias("RELienTypeCode"),
            lit(None).alias("RELienType"),
            prproperty_lien.reason_code.alias("RELienReasonCode"),
            lit(None).alias("RELienReason"),
            prproperty_lien.initial_order.alias("RELienInitialOrder"),
            prproperty_lien.current_order.alias("RELienCurrentOrder"),
            prproperty_lien.creditor_id.cast("string").alias("RELienCreditorKey"),
            lit(None).cast("String").alias("RELienCreditorName"),
            lit(None).cast("String").alias("RELienCreditorType"),
            prproperty_lien.mortgage_type.alias("RELienMortgageTypeCode"),
            lit(None).alias("RELienMortgageType"),
            prproperty_lien.amount.alias("RELienMortgageAmount"),
            prproperty_lien.curr_code.alias("RELienCurrency"),
            prproperty_lien.comments.alias("RELienComments"),
            prproperty_owner.businessdate.alias("BusinessDate"),
        )

        # ================== Joins for temp tables ==================
        thesis_prpropertyownertemp_thesis_prpropertylientemp_join_condition = (
            thesis_prpropertyownertemp["ID"] == thesis_prpropertylientemp["ID"]
        )

        target_table = thesis_prpropertyownertemp.join(
            thesis_prpropertylientemp, 
            thesis_prpropertyownertemp_thesis_prpropertylientemp_join_condition, 
            "left"
        )

        thesis_prpropertyownertemp_primary_key_fields = [
            thesis_prpropertyownertemp.id,
            thesis_prpropertylientemp.id,
            lit("thesis_RES"),
        ]

        RealEstateBuildingKey_primary_key_fields = [
            thesis_prpropertyownertemp.id,
            thesis_prpropertylientemp.id,
        ]

        target_table_thesis_res = (
            target_table.withColumn(
                "RealEstateInfoId",
                sha2(concat_ws(";", *thesis_prpropertyownertemp_primary_key_fields), 256),
            )
            .withColumn(
                "RealEstateInfoKey",
                concat_ws(";", *RealEstateBuildingKey_primary_key_fields),
            )
        )

        # # ================== Selection of Final Columns ==================
        result_df_thesis_res = target_table_thesis_res.select(
            target_table_thesis_res.RealEstateInfoId,
            lit("thesis_RES").alias("SourceSystem"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens")).alias(
                "IngestionTimestamp"
            ),
            lit(None).cast("String").alias("BatchId"),
            target_table_thesis_res.RealEstateInfoKey.cast("string"),
            thesis_prpropertyownertemp.property_temp_id.cast("String").alias("RealEstateKey"),
            thesis_prpropertyownertemp.person_id.cast("String").alias("OwnerPartyKey"),
            thesis_prpropertyownertemp.id.cast("string").alias("OwnershipKey"),
            thesis_prpropertyownertemp.temp_ownership_id.cast("string").alias(
                "OwnershipReferenceKey"
            ),
            thesis_prpropertyownertemp.ownership_id.alias("OwnershipCode"),
            thesis_prpropertyownertemp.realtitle_id.alias("OwnershipTypeCode"),
            thesis_prpropertyownertemp.realtitle_id.alias("OwnershipType"),
            thesis_prpropertyownertemp.percentage.alias("OwnershipPercentage"),
            lit(None).alias("OwnershipStatusCode"),
            lit(None).alias("OwnershipStatus"),
            lit(None).alias("OwnershipStatusDate"),
            thesis_prpropertyownertemp.notary_date.cast("date").alias("NotaryDate"),
            thesis_prpropertyownertemp.notary_no.alias("NotaryNumber"),
            thesis_prpropertylientemp.id.cast("string").alias("RELienKey"),
            thesis_prpropertylientemp.lien_id.cast("string").alias("RELienReferenceKey"),
            lit(None).alias("RELienIsActiveFlag"),
            lit(None).alias("RELienStatusCode"),
            lit(None).alias("RELienStatus"),
            lit(None).alias("RELienActive"),
            thesis_prpropertylientemp.subcat_code.alias("RELienTypeCode"),
            thesis_prpropertylientemp.subcat_code.alias("RELienType"),
            lit(None).alias("RELienReasonCode"),
            lit(None).alias("RELienReason"),
            thesis_prpropertylientemp.initial_order.alias("RELienInitialOrder"),
            lit(None).alias("RELienCurrentOrder"),
            thesis_prpropertylientemp.creditor_id.cast("string").alias("RELienCreditorKey"),
            lit(None).cast("String").alias("RELienCreditorName"),
            lit(None).cast("String").alias("RELienCreditorType"),
            thesis_prpropertylientemp.mortgage_type.alias("RELienMortgageTypeCode"),
            thesis_prpropertylientemp.mortgage_type.alias("RELienMortgageType"),
            thesis_prpropertylientemp.amount.alias("RELienMortgageAmount"),
            thesis_prpropertylientemp.curr_code.alias("RELienCurrency"),
            thesis_prpropertylientemp.comments.alias("RELienComments"),
            thesis_prpropertyownertemp.BusinessDate.alias("BusinessDate"),
        )

        # ================== Union ==================

        # Union the DataFrames
        final_real_estate_info_df = result_df_thesis.union(
            result_df_thesis_res
        )

        # ================== Append ==================

        final_real_estate_info_df.write.format("delta").clusterBy("BusinessDate").mode("append").saveAsTable(
            curated_table_path
        )
    else:
        null_args = [
            arg_name
            for arg_name, arg in args
            if arg
            in [
                curated_zone,
                catalog_landed,
                reference_zone,
                run_date,
                schema,
                env_prefix,
            ]
            and arg is None
        ]

        for arg_name in null_args:
            logging.error(f"No {arg_name} specified!")

    logging.info("We did it!")


In [0]:
load_curated_real_estate_info_thesis(spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix)